Generate cosine similarity to adult mean for both full and half-time drawings. Full drawing embeddings are already column-wise z-scored (from `03_draw_generate_clip_embedding`); half-time embeddings are z-scored here. Output: `data/preprocessed/draw_cossim_adults.csv` with columns `participant_id, age_months, target_category, draw_cossim_adults, draw_half_cossim_adults`.

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
import ast

In [4]:
SEA_ANIMALS = ["crab", "octopus", "seahorse", "shark", "whale", "turtle"]

FULL_EMBED_CSV = "../data/embeddings/clip_sketch_embeddings/image_embeddings/draw_embeddings_with_metadata.csv"
HALF_EMBED_CSV = "../data/embeddings/half_time_sketch_embeddings/half_time_clip_embeddings.csv"
OUT_CSV = "../data/preprocessed/draw_cossim_adults.csv"

In [5]:
def parse_embedding(x):
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, list):
        return np.array(x, dtype=float)
    if pd.isna(x):
        return np.nan
    return np.array(ast.literal_eval(x), dtype=float)

def mean_embedding(emb_series):
    arr = np.vstack(emb_series.to_list())
    return arr.mean(axis=0)

def column_z_score(emb_series):
    valid_mask = emb_series.notna().values
    arr = emb_series.values.copy()
    matrix = np.vstack(arr[valid_mask]).astype(float)
    matrix = (matrix - matrix.mean(axis=0)) / matrix.std(axis=0)
    for i, pos in enumerate(np.where(valid_mask)[0]):
        arr[pos] = matrix[i]
    return pd.Series(arr, index=emb_series.index)

In [6]:
# FULL DRAWINGS — already column-wise z-scored in 03_draw_generate_clip_embedding
full_df = pd.read_csv(FULL_EMBED_CSV)
full_df["embedding"] = full_df["embedding"].apply(parse_embedding)
full_df = full_df[full_df["target_category"].isin(SEA_ANIMALS)].copy()
full_df["is_adult"] = full_df["participant_id"].str.upper().str.startswith("AD")
print("Full drawings (sea animals):", full_df.shape)
print(full_df["is_adult"].value_counts())

Full drawings (sea animals): (638, 6)
is_adult
False    518
True     120
Name: count, dtype: int64


In [7]:
adult_means_full = (
    full_df[full_df["is_adult"]]
    .groupby("target_category")["embedding"]
    .apply(mean_embedding)
)
full_df["adult_mean"] = full_df["target_category"].map(adult_means_full)

mask = (
    full_df["adult_mean"].notna()
    & full_df["embedding"].apply(lambda x: isinstance(x, np.ndarray) and len(x) == 512)
)
E = np.vstack(full_df.loc[mask, "embedding"].to_list()).astype(float)
A = np.vstack(full_df.loc[mask, "adult_mean"].to_list()).astype(float)

num = np.sum(E * A, axis=1)
den = np.linalg.norm(E, axis=1) * np.linalg.norm(A, axis=1)
full_df.loc[mask, "draw_cossim_adults"] = num / den

# Drop adults — they were used to build the mean, not the output
full_df = full_df[~full_df["is_adult"]].copy()
print(full_df["draw_cossim_adults"].describe())

count    518.000000
mean       0.282857
std        0.297036
min       -0.355258
25%        0.039315
50%        0.292083
75%        0.508518
max        0.880261
Name: draw_cossim_adults, dtype: float64


## Half-time drawings

Half-time embeddings have not been z-scored yet — z-scored column-wise here to match full-drawing normalization.

In [8]:
# HALF-TIME DRAWINGS — z-score column-wise here
half_df = pd.read_csv(HALF_EMBED_CSV)
half_df["half_time_embedding"] = half_df["half_time_embedding"].apply(parse_embedding)

# Strip "a " / "an " prefix so categories match the full-drawing vocabulary
half_df["target_category"] = half_df["category"].str.replace(r"^(a|an)\s+", "", regex=True)
half_df = half_df[half_df["target_category"].isin(SEA_ANIMALS)].copy()

# Exclude pilot participants: keep only AD001-AD020 and BD025-BD125
half_df["is_adult"] = half_df["participant_id"].str.upper().str.startswith("AD")
half_df["_bd_num"] = pd.to_numeric(
    half_df["participant_id"].str.extract(r"^[Bb][Dd](\d+)$")[0], errors="coerce"
)
half_df = half_df[half_df["is_adult"] | (half_df["_bd_num"] >= 25)].drop(columns="_bd_num")

half_df["half_time_embedding"] = column_z_score(half_df["half_time_embedding"])
print("Half-time drawings (sea animals, non-pilot):", half_df.shape)
print(half_df["is_adult"].value_counts())

Half-time drawings (sea animals, non-pilot): (639, 11)
is_adult
False    519
True     120
Name: count, dtype: int64


In [9]:
adult_means_half = (
    half_df[half_df["is_adult"]]
    .groupby("target_category")["half_time_embedding"]
    .apply(mean_embedding)
)
half_df["adult_mean"] = half_df["target_category"].map(adult_means_half)

mask = (
    half_df["adult_mean"].notna()
    & half_df["half_time_embedding"].apply(lambda x: isinstance(x, np.ndarray) and len(x) == 512)
)
E = np.vstack(half_df.loc[mask, "half_time_embedding"].to_list()).astype(float)
A = np.vstack(half_df.loc[mask, "adult_mean"].to_list()).astype(float)

num = np.sum(E * A, axis=1)
den = np.linalg.norm(E, axis=1) * np.linalg.norm(A, axis=1)
half_df.loc[mask, "draw_half_cossim_adults"] = num / den

# Drop adults — they were used to build the mean, not the output
half_df = half_df[~half_df["is_adult"]].copy()
print(half_df["draw_half_cossim_adults"].describe())

count    519.000000
mean       0.149333
std        0.260675
min       -0.353490
25%       -0.063352
50%        0.104448
75%        0.325927
max        0.850365
Name: draw_half_cossim_adults, dtype: float64


In [10]:
result = full_df[["participant_id", "age_months", "target_category", "draw_cossim_adults"]].merge(
    half_df[["participant_id", "target_category", "draw_half_cossim_adults"]],
    on=["participant_id", "target_category"],
    how="outer"
)
print("Merged shape:", result.shape)
result.head()

Merged shape: (519, 5)


,participant_id,age_months,target_category,draw_cossim_adults,draw_half_cossim_adults
0,BD025,134.0,crab,0.121349,-0.012768
1,BD025,134.0,octopus,0.473362,0.681731
2,BD025,134.0,seahorse,0.581902,0.543585
3,BD025,134.0,shark,0.795955,0.256982
4,BD025,134.0,turtle,0.367491,0.217603


In [11]:
print(result.groupby("target_category")[["draw_cossim_adults", "draw_half_cossim_adults"]].describe().round(3))

Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT_CSV, index=False)
print(f"\nSaved {result.shape[0]} rows → {OUT_CSV}")

                draw_cossim_adults                                            \
                             count   mean    std    min    25%    50%    75%   
target_category                                                                
crab                          87.0  0.248  0.253 -0.223  0.066  0.253  0.424   
octopus                       89.0  0.310  0.290 -0.252  0.075  0.359  0.521   
seahorse                      86.0  0.295  0.272 -0.273  0.086  0.288  0.543   
shark                         84.0  0.308  0.354 -0.288 -0.020  0.339  0.602   
turtle                        86.0  0.330  0.314 -0.204  0.039  0.345  0.612   
whale                         86.0  0.205  0.281 -0.355 -0.030  0.202  0.408   

                       draw_half_cossim_adults                              \
                   max                   count   mean    std    min    25%   
target_category                                                              
crab             0.804                    87.